# Colab S1 — El experimento repetido: de dónde sale la distribución de χ²

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Complemento de las Clases 2, 3 y 6 — material optativo

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/S1_El_experimento_repetido_y_la_distribucion_de_chi2.ipynb)

Cuando ajustás y te da $\chi^2_\nu = 1{,}7$, la pregunta natural es *¿eso está bien o está mal?*. Este cuaderno la contesta de la única manera convincente: **haciendo el experimento cinco mil veces** y mirando qué valores salen.

No hace falta ninguna fórmula nueva. Todo lo que sigue es simulación: generamos datos con ruido conocido, los analizamos exactamente como analizamos los reales, y vemos qué distribución tienen los resultados.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. Un experimento, simulado

Fabricamos un experimento del que **conocemos la verdad**: el modelo, los
parámetros y el ruido. Eso es justamente lo que nunca tenemos en la mesada,
y es lo que permite ver si nuestros métodos de análisis funcionan.

In [ ]:
generador = np.random.default_rng(2026)

a_real, b_real = 2.500, 1.000
sigma_real = 0.20
N = 12

x = np.linspace(0, 10, N)


def experimento(gen):
    # Un experimento completo: N mediciones con ruido gaussiano conocido.
    return a_real*x + b_real + gen.normal(0, sigma_real, size=N)


y = experimento(generador)

fig, ax = plt.subplots()
ax.errorbar(x, y, yerr=sigma_real, fmt="o", capsize=3, label="una realización")
ax.plot(x, a_real*x + b_real, "crimson", lw=1.5, label="la verdad")
ax.set_xlabel("x (u. arb.)")
ax.set_ylabel("y (u. arb.)")
ax.legend()
plt.show()

### 2. El ruido es lo que dijimos que era

Antes de seguir, verifiquemos que el ruido que metimos es el que sale. Los
residuos respecto de la **verdad** (no del ajuste) tienen que distribuirse
como una gaussiana de ancho $\sigma$.

In [ ]:
muchos_residuos = np.concatenate(
    [experimento(generador) - (a_real*x + b_real) for _ in range(2000)])

media, s, sem = lab.estadisticos(muchos_residuos)
print(f"sigma que pusimos: {sigma_real}")

fig, ax = plt.subplots()
ax.hist(muchos_residuos, bins=80, density=True, edgecolor="none")
u = np.linspace(-1, 1, 300)
ax.plot(u, np.exp(-u**2/(2*sigma_real**2))/(sigma_real*np.sqrt(2*np.pi)),
        "crimson", lw=2)
ax.set_xlabel("Residuo (u. arb.)")
ax.set_ylabel("Densidad")
plt.show()

### 3. La distribución del promedio: el SEM, verificado

En la Clase 2 dijimos que la incerteza del promedio es $s/\sqrt{N}$ y que
representa *cuánto se dispersaría el promedio si repitieras el experimento*.
Nadie repite el experimento mil veces. Acá sí.

In [ ]:
M = 5000
promedios = np.array([experimento(generador).mean() for _ in range(M)])

dispersion_real = np.std(promedios, ddof=1)
prediccion = sigma_real/np.sqrt(N)

print(f"dispersión observada de {M} promedios : {dispersion_real:.5f}")
print(f"predicción sigma/sqrt(N)             : {prediccion:.5f}")
print(f"coinciden dentro del {100*abs(dispersion_real/prediccion - 1):.1f} %")

fig, ax = plt.subplots()
ax.hist(promedios, bins=60, density=True, edgecolor="none")
ax.set_xlabel("Promedio de cada experimento (u. arb.)")
ax.set_ylabel("Densidad")
plt.show()

Ahí está el significado operativo del SEM, sin ninguna fórmula: es el ancho
del histograma de los promedios que obtendrías repitiendo el experimento
completo. Vos hacés **un** experimento y sacás **un** punto de ese
histograma; el SEM te dice cuán ancho es el histograma del que salió.

### 4. El χ² de un ajuste no es un número: es una muestra

Ahora la parte central. Para cada experimento simulado ajustamos la recta,
calculamos su $\chi^2$ y lo guardamos. Con $N = 12$ puntos y 2 parámetros,
$\nu = 10$.

In [ ]:
from scipy.optimize import curve_fit
from scipy import stats


def recta(x, a, b):
    return a*x + b


def un_experimento_completo(gen, sigma_usada=None, modelo=recta, p0=None):
    # Genera datos, ajusta y devuelve chi2 y los parámetros.
    y = experimento(gen)
    s = sigma_usada if sigma_usada is not None else sigma_real
    err = np.full(N, s)
    popt, _ = curve_fit(modelo, x, y, sigma=err, absolute_sigma=True, p0=p0)
    chi2 = np.sum(((y - modelo(x, *popt))/err)**2)
    return chi2, popt


chi2s = np.array([un_experimento_completo(generador)[0] for _ in range(M)])
nu = N - 2

print(f"ν = {nu}")
print(f"promedio de los χ²   : {chi2s.mean():.2f}   (teórico: {nu})")
print(f"promedio de los χ²_ν : {(chi2s/nu).mean():.3f}   (teórico: 1)")
print(f"dispersión de los χ²_ν: {np.std(chi2s/nu, ddof=1):.3f} "
      f"  (teórico: {np.sqrt(2/nu):.3f})")

In [ ]:
fig, ax = plt.subplots()
ax.hist(chi2s/nu, bins=70, density=True, edgecolor="none",
        label=f"{M} experimentos simulados")
u = np.linspace(0.01, 4, 400)
ax.plot(u, stats.chi2.pdf(u*nu, nu)*nu, "crimson", lw=2,
        label=f"distribución χ² teórica (ν = {nu})")
ax.axvline(1, color="k", ls="--", lw=1)
ax.set_xlabel(r"$\chi^2_\nu$")
ax.set_ylabel("Densidad")
ax.legend()
plt.show()

Éste es el gráfico que hay que tener en la cabeza cada vez que aparece un
$\chi^2_\nu$.

Con el modelo **correcto** y las barras **correctas**, el $\chi^2_\nu$ no da
1: da un número sacado de esa distribución, que tiene una cola derecha
apreciable. Con $\nu = 10$ es perfectamente normal obtener 1,5. Es la
respuesta a la pregunta que todos los años aparece en la mesada: *me dio 1,7,
¿está mal?*.

In [ ]:
for umbral in [1.0, 1.5, 2.0, 2.5]:
    frac = np.mean(chi2s/nu > umbral)
    print(f"P(χ²_ν > {umbral:.1f}) = {100*frac:5.2f} %      "
          f"(teórico: {100*stats.chi2.sf(umbral*nu, nu):5.2f} %)")

Y acá se ve por qué el umbral depende de $\nu$: la distribución se angosta
como $\sqrt{2/\nu}$.

In [ ]:
fig, ax = plt.subplots()
u = np.linspace(0.01, 3, 400)
for nu_i in [3, 10, 30, 100]:
    ax.plot(u, stats.chi2.pdf(u*nu_i, nu_i)*nu_i, lw=1.8, label=f"ν = {nu_i}")
ax.axvline(1, color="k", ls="--", lw=1)
ax.set_xlabel(r"$\chi^2_\nu$")
ax.set_ylabel("Densidad")
ax.legend()
plt.show()

print("Con ν = 3, un χ²_ν de 1,8 es rutina.")
print("Con ν = 100, un χ²_ν de 1,8 no pasa nunca por azar.")

### 5. El p-valor es uniforme (y por eso sirve)

Si el modelo es correcto y las barras están bien, el p-valor de un ajuste
está distribuido **uniformemente entre 0 y 1**. Eso es lo que lo convierte
en una regla de decisión: "rechazo si p < 0,01" rechaza exactamente el 1 %
de los ajustes buenos.

In [ ]:
pvalores = stats.chi2.sf(chi2s, nu)

fig, ax = plt.subplots()
ax.hist(pvalores, bins=25, density=True, edgecolor="black")
ax.axhline(1, color="crimson", lw=2)
ax.set_xlabel("p-valor")
ax.set_ylabel("Densidad")
ax.set_ylim(0, 2)
plt.show()

print(f"fracción con p < 0,01 : {100*np.mean(pvalores < 0.01):.2f} %")
print(f"fracción con p > 0,99 : {100*np.mean(pvalores > 0.99):.2f} %")
print()
print("Los dos extremos son igual de improbables. Por eso un p-valor de")
print("0,999 es tan sospechoso como uno de 0,001: significa que los datos")
print("se parecen demasiado al modelo para el ruido que declaraste.")

### 6. ¿Qué pasa si el modelo está mal?

Ajustamos los mismos datos —generados con una recta— usando una constante.
El modelo es incorrecto por construcción.

In [ ]:
def constante(x, c):
    return c + 0*x


chi2_malo = np.array([un_experimento_completo(generador, modelo=constante,
                                              p0=[10.0])[0]
                      for _ in range(1000)])
nu_malo = N - 1

fig, ax = plt.subplots()
ax.hist(chi2s/nu, bins=60, density=True, alpha=0.7, label="modelo correcto")
ax.hist(chi2_malo/nu_malo, bins=60, density=True, alpha=0.7,
        label="modelo incorrecto")
ax.set_xlabel(r"$\chi^2_\nu$")
ax.set_ylabel("Densidad")
ax.set_xscale("log")
ax.legend()
plt.show()

print(f"χ²_ν típico con el modelo correcto  : {np.median(chi2s/nu):.2f}")
print(f"χ²_ν típico con el modelo incorrecto: "
      f"{np.median(chi2_malo/nu_malo):.1f}")

Las dos distribuciones no se tocan. Ése es el sentido de "el $\chi^2$
detecta modelos mal especificados": no es que el número sea feo, es que cae
en una región donde el modelo correcto **nunca** cae.

### 7. ¿Y si las barras de error están mal?

Éste es el caso más frecuente en la práctica, y el más difícil de
diagnosticar, porque el modelo puede estar perfecto.

In [ ]:
print("      sigma declarada     χ²_ν mediano     lectura")
for factor in [0.5, 0.8, 1.0, 1.25, 2.0]:
    c2 = np.array([un_experimento_completo(generador,
                                           sigma_usada=factor*sigma_real)[0]
                   for _ in range(600)])/nu
    if np.median(c2) > 1.5:
        lectura = "subestimé las barras"
    elif np.median(c2) < 0.7:
        lectura = "sobreestimé las barras"
    else:
        lectura = "coherente"
    print(f"      {factor:4.2f} × real        {np.median(c2):6.2f}         {lectura}")

El $\chi^2_\nu$ escala como $1/\text{factor}^2$: declarar la mitad del error
real cuadruplica el $\chi^2_\nu$.

**Cómo se distingue de un modelo malo:** por los residuos. Barras
subestimadas dan residuos grandes pero **sin estructura**; un modelo malo da
residuos **con forma**. Ésa es la razón de que el panel de residuos sea
obligatorio y no un adorno.

### 8. Qué hacer con tu único χ²

Vos vas a tener un solo experimento y un solo número. La secuencia de
lectura, con todo lo anterior en la mano:

1. Calculá $\chi^2_\nu$ **y** el p-valor. Nunca el primero solo.
2. Si $0{,}01 < p < 0{,}99$: no hay evidencia contra tu modelo. No digas que
   "se comprobó" el modelo; decí que los datos son consistentes con él.
3. Si $p < 0{,}01$: mirá los residuos. ¿Tienen forma? Es el modelo. ¿Son
   ruido puro pero grandes? Son las barras.
4. Si $p > 0{,}99$: casi seguro sobreestimaste las incertezas. Revisá si no
   estás contando dos veces la misma fuente de error.
5. En todos los casos, reportá $\chi^2_\nu$, $\nu$ y $p$. Los tres. Un
   $\chi^2_\nu$ sin $\nu$ no se puede interpretar.

### 9. Ejercicios

1. Cambiá $N$ de 12 a 50 y mirá cómo se angosta la distribución de
   $\chi^2_\nu$. Verificá que el ancho sigue $\sqrt{2/\nu}$.
2. Simulá el caso en que el ruido **no** es gaussiano (usá
   `generador.uniform` con la misma varianza). ¿Sigue valiendo la
   distribución de $\chi^2$? ¿A partir de qué $N$?
3. Simulá barras de error **heterogéneas** (por ejemplo, proporcionales a
   $y$) y verificá que el ajuste ponderado recupera los parámetros verdaderos
   mientras que el no ponderado los sesga. Cuantificá el sesgo.
4. Guardá también los parámetros ajustados de los 5000 experimentos y
   comparalos con el error que devuelve `pcov` en uno solo. Ésa es la
   verificación de que `absolute_sigma=True` da la incerteza correcta: hacela
   también con `False` y mirá la diferencia.

In [ ]:
# Espacio de trabajo para los ejercicios.